In [9]:
!git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git


fatal: destination path 'PlantVillage-Dataset' already exists and is not an empty directory.


In [10]:
!ls PlantVillage-Dataset/raw/color | grep Potato


Potato___Early_blight
Potato___healthy
Potato___Late_blight


In [11]:
import torch
from torchvision import datasets, transforms

# 1. Define a transform pipeline: resize images to a fixed size,
#    convert to a tensor, and normalize pixel values.
transform = transforms.Compose([
    transforms.Resize((128,128)),   # what size? (hint: 96 or 128 is fine for a first try)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # hint: look up "ImageNet mean std" if unsure
])

# 2. Point ImageFolder at the directory containing your 3 class folders
dataset = datasets.ImageFolder(root="PlantVillage-Dataset/raw/color", transform=transform)

# 3. Print how many images got loaded, and what the classes are
print(len(dataset))
print(dataset.classes)

2152
['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']


In [12]:
import os, shutil
base = "PlantVillage-Dataset/raw/color"
keep = {"Potato___healthy", "Potato___Early_blight", "Potato___Late_blight"}
for folder in os.listdir(base):
    if folder not in keep:
        shutil.rmtree(os.path.join(base, folder))

In [13]:
from torch.utils.data import DataLoader, random_split

val_size = int(0.2 * len(dataset))       # 20% held out for validation
train_size = len(dataset) - val_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

print(f"Train: {train_size}, Val: {val_size}")

Train: 1722, Val: 430


In [14]:
import torch.nn as nn

class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SmallCNN(num_classes=len(dataset.classes))

In [15]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    torch.set_grad_enabled(train)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train:
            optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

for epoch in range(8):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1} | train_acc={tr_acc:.3f} | val_acc={val_acc:.3f}")

Epoch 1 | train_acc=0.876 | val_acc=0.942
Epoch 2 | train_acc=0.948 | val_acc=0.933
Epoch 3 | train_acc=0.952 | val_acc=0.965
Epoch 4 | train_acc=0.965 | val_acc=0.967
Epoch 5 | train_acc=0.959 | val_acc=0.972
Epoch 6 | train_acc=0.967 | val_acc=0.979
Epoch 7 | train_acc=0.977 | val_acc=0.972
Epoch 8 | train_acc=0.969 | val_acc=0.970


In [16]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        out = model(x)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(y.numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=dataset.classes))

[[196   5   0]
 [  2 196   3]
 [  0   3  25]]
                       precision    recall  f1-score   support

Potato___Early_blight       0.99      0.98      0.98       201
 Potato___Late_blight       0.96      0.98      0.97       201
     Potato___healthy       0.89      0.89      0.89        28

             accuracy                           0.97       430
            macro avg       0.95      0.95      0.95       430
         weighted avg       0.97      0.97      0.97       430



In [18]:
torch.save(model.state_dict(), "leaf_classifier.pth")
